# Analyzing RCT data with Precision Adjustment

In [33]:
import pandas as pd
import numpy as np
import patsy
import statsmodels.formula.api as smf
import statsmodels.api as sm

## Data

In this lab, we analyze the Pennsylvania re-employment bonus experiment, which was previously studied in "Sequential testing of duration data: the case of the Pennsylvania ‘reemployment bonus’ experiment" (Bilias, 2000), among others. These experiments were conducted in the 1980s by the U.S. Department of Labor to test the incentive effects of alternative compensation schemes for unemployment insurance (UI). In these experiments, UI claimants were randomly assigned either to a control group or one of six treatment groups. Here we focus on treatment group 4, but feel free to explore other treatment groups. In the control group the current rules of UI applied. Individuals in the treatment groups were offered a cash bonus if they found a job within some pre-specified period of time (qualification period), provided that the job was retained for a specified duration. The treatments differed in the level of the bonus, the length of the qualification period, and whether the bonus was declining over time in the qualification period; see http://qed.econ.queensu.ca/jae/2000-v15.6/bilias/readme.b.txt for further details on data.
  

In [34]:
data = pd.read_csv("https://raw.githubusercontent.com/VC2015/DMLonGitHub/master/penn_jae.dat", sep='\\s+')
n, p = data.shape
data = data[data["tg"].isin([0, 4])]

In [35]:
data["T4"] = np.where(data["tg"] == 4, 1, 0)
data["T4"].value_counts()

T4
0    3354
1    1745
Name: count, dtype: int64

In [36]:
data.describe()

,abdt,tg,inuidur1,inuidur2,female,black,hispanic,othrace,dep,q1,...,q6,recall,agelt35,agegt54,durable,nondurable,lusd,husd,muld,T4
count,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,...,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000,5099.000000
mean,10695.416356,1.368896,13.052952,12.281232,0.404001,0.121985,0.032555,0.007256,0.439694,0.012748,...,0.062954,0.110414,0.545009,0.109433,0.148068,0.109237,0.261032,0.218670,0.444205,0.342224
std,111.180503,1.898003,10.565602,10.362143,0.490746,0.327300,0.177487,0.084883,0.757622,0.112194,...,0.242903,0.313436,0.498019,0.312213,0.355202,0.311967,0.439240,0.413385,0.496926,0.474501
min,10404.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10600.000000,0.000000,3.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,10698.000000,0.000000,11.000000,10.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,10796.000000,4.000000,25.000000,23.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
max,10880.000000,4.000000,52.000000,52.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Model
To evaluate the impact of the treatments on unemployment duration, we consider the linear regression model:

$$
Y =  D \beta_1 + W'\beta_2 + \varepsilon, \quad E \varepsilon (D,W')' = 0,
$$

where $Y$ is  the  log of duration of unemployment, $D$ is a treatment  indicator,  and $W$ is a set of controls including age group dummies, gender, race, number of dependents, quarter of the experiment, location within the state, existence of recall expectations, and type of occupation.   Here $\beta_1$ is the ATE, assuming the RCT assumptions hold. Within an RCT, the projection coefficient $\beta_1$ has
the interpretation of the causal effect of the treatment on
the average outcome. We thus refer to $\beta_1$ as the average
treatment effect (ATE). Note that the covariates, here are
independent of the treatment $D$, so we can identify $\beta_1$ by
just linear regression of $Y$ on $D$, without adding covariates.
However, we do add covariates in an effort to improve the
precision of our estimates of the average treatment effect.


We also consider an interactive regression model:

$$
Y =  D \alpha_1 + D W' \alpha_2 + W'\beta_2 + \varepsilon, \quad E \varepsilon (D,W', DW')' = 0,
$$
where $W$'s are demeaned (apart from the intercept), so that $\alpha_1$ is the ATE, assuming the RCT assumptions hold. Unlike the model without interactions, we are guaranteed to improve precision by considering the interactive model.

### Analysis

We consider

*  classical 2-sample approach, no adjustment (CL)
*  classical linear regression adjustment (CRA)
*  interactive regression adjusment (IRA)

# Carry out covariate balance check


We first look at the coefficients individually with a $t$-test, and then we adjust the $p$-values to control for family-wise error.

The regression below is done using "type='HC1'" which computes the correct Eicker-Huber-White standard errors, instead of the classical non-robust formula based on homoscedasticity.

In [37]:
formula = ("0 ~ "
           "(female + black + othrace + C(dep) + q2 + q3 + q4 + q5 + q6 "
           "+ agelt35 + agegt54 + durable + lusd + husd)**2")
X = patsy.dmatrix(formula, data=data, return_type='dataframe')
X.shape

(5099, 120)

There is strong multicollinearity in the design matrix. In order to address this, we drop columns by thresholding using the QR decomposition.

In [38]:
def qr_decomposition(x):
    # Get a set of columns with no linear dependence for smallest subset of observations
    Q, Rx = np.linalg.qr(x)
    ex = np.abs(np.diag(Rx))
    keep = np.where(ex > 1e-6)[0]
    xS = x.iloc[:, keep]

    return xS


xS = qr_decomposition(X)
xS.shape

(5099, 103)

In [39]:
# individual t-tests
covariate_balance = sm.OLS(data["T4"], xS)
cb_fit = covariate_balance.fit(cov_type="HC1", use_t=True)
cb_fit.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     T4   R-squared:                       0.029
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     29.07
Date:                Fri, 20 Feb 2026   Prob (F-statistic):               0.00
Time:                        21:02:21   Log-Likelihood:                -3358.5
No. Observations:                5099   AIC:                             6923.
Df Residuals:                    4996   BIC:                             7596.
Df Model:                         102                                         
Covariance Type:                  HC1                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               0.3215      0.161      1.993      0.046       0.005       0.638
C(dep)[T.1]            -0.0736      0.201     -0.366      0.714      -0.468       0.320
C(dep)[T.2]            -0.1085      0.158     -0.689      0.491      -0.417       0.200
female                  0.1042      0.136      0.765      0.444      -0.163       0.371
female:C(dep)[T.1]      0.0555      0.046      1.194      0.233      -0.036       0.147
female:C(dep)[T.2]      0.0454      0.041      1.121      0.262      -0.034       0.125
black                   0.0716      0.083      0.864      0.387      -0.091       0.234
black:C(dep)[T.1]      -0.1173      0.068     -1.725      0.085      -0.251       0.016
black:C(dep)[T.2]      -0.0222      0.063     -0.350      0.726      -0.147       0.102
othrace                 0.0280      0.405      0.069      0.945      -0.766       0.822
othrace:C(dep)[T.1]    -0.8581      0.331     -2.594      0.010      -1.507      -0.210
othrace:C(dep)[T.2]     0.2415      0.133      1.812      0.070      -0.020       0.503
q2                     -0.0267      0.163     -0.164      0.870      -0.345       0.292
C(dep)[T.1]:q2          0.1658      0.201      0.825      0.409      -0.228       0.559
C(dep)[T.2]:q2          0.0875      0.157      0.559      0.576      -0.220       0.395
q3                     -0.0057      0.162     -0.035      0.972      -0.324       0.312
C(dep)[T.1]:q3          0.1210      0.200      0.606      0.545      -0.270       0.512
C(dep)[T.2]:q3          0.1396      0.157      0.891      0.373      -0.168       0.447
q4                      0.0433      0.162      0.267      0.789      -0.275       0.362
C(dep)[T.1]:q4          0.0848      0.199      0.426      0.670      -0.306       0.475
C(dep)[T.2]:q4          0.0910      0.157      0.580      0.562      -0.217       0.399
q5                      0.0939      0.162      0.581      0.561      -0.223       0.411
C(dep)[T.1]:q5          0.1105      0.198      0.557      0.578      -0.278       0.499
C(dep)[T.2]:q5          0.0960      0.155      0.618      0.537      -0.209       0.401
q6                     -0.2216      0.160     -1.386      0.166      -0.535       0.092
C(dep)[T.1]:q6          0.1007      0.209      0.482      0.630      -0.309       0.510
C(dep)[T.2]:q6          0.0484      0.164      0.295      0.768      -0.273       0.370
agelt35                -0.1092      0.133     -0.820      0.412      -0.370       0.152
C(dep)[T.1]:agelt35    -0.0737      0.049     -1.518      0.129      -0.169       0.021
C(dep)[T.2]:agelt35    -0.0244      0.038     -0.650      0.515      -0.098       0.049
agegt54                -0.4367      0.136     -3.215      0.001      -0.703      -0.170
C(dep)[T.1]:agegt54    -0.0806      0.068     -1.180      0.238      -0.214       0.053
C(d

To test balance conditions, we employ the Holm-Bonferroni step-down method. With 100+ hypotheses, the family-wise type I error, or the probability of making at least one type I error treating all hypotheses independently, is close to 1. To control for this, we adjust p-values with the following procedure.

First, set $\alpha=0.05$ and denote the list of $n$ p-values from the regression with the vector $p$.

1. Sort $p$ from smallest to largest, so $p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(n)}$. Denote the corresponding hypothesis for $p_{(i)}$ as $H_{(i)}$.
2. For $i=1,\ldots, n$,
- If $$p_{(i)} > \frac{\alpha}{n-i+1} $$ Break the loop and do not reject any $H_{(j)}$ for $j \geq i$.
- Else reject $H_{(i)}$ if $$p_{(i)} \leq \frac{\alpha}{n-i+1} $$ Increment $i := i+1$.




In [40]:
def holm_bonferroni(p, alpha=0.05):

    n = len(p)
    sig_beta = []

    for i in range(n):
        if np.sort(p)[i] > alpha / (n - i):
            break
        else:
            sig_beta.append(np.argsort(p).iloc[i])
            i += 1

    return sig_beta


print("Significant Coefficients (Indices): ", holm_bonferroni(cb_fit.pvalues, alpha=0.05))

Significant Coefficients (Indices):  [np.int64(83), np.int64(73), np.int64(93), np.int64(78)]


We see that that even though this is a randomized experiment, balance conditions are failed.
<!--
The holm method fails to reject any hypothesis. That is, we fail to reject the hypothesis that any coefficient is zero. Thus, in this randomized experiment, balance conditions are met. -->

# Model Specification

In [41]:
# no adjustment (2-sample approach)
cl = smf.ols("np.log(inuidur1) ~ T4", data)

# adding controls
cra = smf.ols("np.log(inuidur1) ~ T4 + "
              "(female+black+othrace+C(dep)+q2+q3+q4+q5+q6+agelt35+agegt54+durable+lusd+husd)**2",
              data)
# Omitted dummies: q1, nondurable, muld

cl_results = cl.fit(cov_type="HC0")
cl_est = cl_results.params['T4']
cl_se = cl_results.HC0_se['T4']
print(f"ATE: {cl_est:.4f}, (std={cl_se:.4f})")

ATE: -0.0855, (std=0.0358)


In [42]:
cra_results = cra.fit(cov_type="HC0")
cra_est = cra_results.params['T4']
cra_se = cra_results.HC0_se['T4']
print(f"ATE: {cra_est:.4f}, (std={cra_se:.4f})")

ATE: -0.0797, (std=0.0352)


The interactive specification corresponds to the approach introduced in Lin (2013).

In [44]:
# interactive regression model

ira_formula = "0 + (female+black+othrace+C(dep)+q2+q3+q4+q5+q6+agelt35+agegt54+durable+lusd+husd)**2"
X = patsy.dmatrix(ira_formula, data, return_type='dataframe')
X.columns = [f'x{t}' for t in range(X.shape[1])]  # clean column names
X = (X - X.mean(axis=0))  # demean all control covariates

# construct interactions of treatment and (de-meaned covariates, 1)
ira_formula = "T4 * (" + "+".join(X.columns) + ")"
X['T4'] = data['T4']
X = patsy.dmatrix(ira_formula, X, return_type='dataframe')
ira = sm.OLS(np.log(data[["inuidur1"]]), X)
ira_results = ira.fit(cov_type="HC0")
ira_est = ira_results.params['T4']

# because we de-meaned the covariates and interacted them with the treamtent
# we need to account for the error in the means of X, which has a first-order
# effect on the ATE estimate
interaction_cols = list(map(lambda x: x.startswith('T4:'), list(X.columns)))
cate = X.iloc[:, interaction_cols] @ ira_results.params[interaction_cols]
ira_se = np.sqrt(ira_results.HC0_se['T4']**2 + np.var(cate) / X.shape[0])

print(f"ATE: {ira_est:.4f}, (std={ira_se:.4f})")

ATE: -195221331.3866, (std=244527348.3408)


In [45]:
print("Python version:", sys.version)
print("patsy version:", patsy.__version__)

Python version: 3.13.7 (main, Aug 14 2025, 11:12:11) [Clang 17.0.0 (clang-1700.0.13.3)]
patsy version: 1.0.2


# Partialling Out with the Lasso
Next we try out partialling out with theoretically justified lasso. We use "plug-in" tuning with a theoretically valid choice of penalty $\lambda = 2 \cdot c \hat{\sigma} \sqrt{n} \Phi^{-1}(1-\alpha/2p)$, where $c>1$ and $1-\alpha$ is a confidence level, and $\Phi^{-1}$ denotes the quantile function. See book for more details.

We pull an analogue of R's rlasso. We find a Python code that tries to replicate the main function of hdm r-package. It was made by [Max Huppertz](https://maxhuppertz.github.io/code/). His library is this [repository](https://github.com/maxhuppertz/hdmpy). If not using colab, download its repository and copy this folder to your site-packages folder. In my case it is located here ***C:\Python\Python38\Lib\site-packages*** . We need to install this package ***pip install multiprocess***.

In [12]:
!pip install multiprocess
!pip install pyreadr
!git clone https://github.com/maxhuppertz/hdmpy.git

Cloning into 'hdmpy'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 89 (delta 47), reused 63 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 28.88 KiB | 4.81 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [13]:
import sys
sys.path.insert(1, "./hdmpy")

In [18]:
# We wrap the package so that it has the familiar sklearn API
import hdmpy
from sklearn.base import BaseEstimator


class RLasso(BaseEstimator):

    def __init__(self, *, post=True):
        self.post = post

    def fit(self, X, y):
        self.rlasso_ = hdmpy.rlasso(X, y, post=self.post)
        return self

    def predict(self, X):
        return np.array(X) @ np.array(self.rlasso_.est['beta']).flatten() + np.array(self.rlasso_.est['intercept'])

    @property
    def coef_(self,):
        return np.array(self.rlasso_.est['beta']).flatten()


def lasso_model():
    return RLasso(post=False)

## Non-Interactive Model

In [19]:
y = np.log(data["inuidur1"])
D = data['T4']
ira_formula = "0 + (female+black+othrace+C(dep)+q2+q3+q4+q5+q6+agelt35+agegt54+durable+lusd+husd)**2"
X = patsy.dmatrix(ira_formula, data, return_type='dataframe')
X = X.loc[:, X.std() > 0]  # drop constant columns

Dres = D - np.mean(D)
yres = y - lasso_model().fit(X, y).predict(X)

cra_lasso_results = sm.OLS(yres, Dres).fit(cov_type='HC0')
cra_lasso_est = cra_lasso_results.params['T4']
cra_lasso_se = cra_lasso_results.HC0_se['T4']
print(f"ATE: {cra_lasso_est:.4f}, (std={cra_lasso_se:.4f})")

ATE: -0.0794, (std=0.0355)


## Interactive Model

In [16]:
X.columns = [f'x{t}' for t in range(X.shape[1])]  # clean column names
X = (X - X.mean(axis=0))  # demean all control covariates

# construct interactions of treatment and (de-meaned covariates, 1)
ira_formula = "T4 * (" + "+".join(X.columns) + ")"
X['T4'] = data['T4']
X = patsy.dmatrix(ira_formula, X, return_type='dataframe')
# drop treatment column
X = X.drop('T4', axis=1)
X = X.loc[:, X.std() > 0]  # drop constant columns

Dres = D - np.mean(D)
modely = lasso_model().fit(X, y)
yres = y - modely.predict(X)


ira_lasso_results = sm.OLS(yres, Dres).fit(cov_type='HC0')
ira_lasso_est = ira_lasso_results.params['T4']

# because we de-meaned the covariates and interacted them with the treamtent
# we need to account for the error in the means of X, which has a first-order
# effect on the ATE estimate
interaction_cols = list(map(lambda x: x.startswith('T4:'), list(X.columns)))
cate = X.iloc[:, interaction_cols] @ modely.coef_[interaction_cols]
ira_lasso_se = np.sqrt(ira_lasso_results.HC0_se['T4']**2 + np.var(cate) / X.shape[0])

# Overall Results

In [17]:
results = {
    "CL": [cl_est, cl_se],
    "CRA": [cra_est, cra_se],
    "IRA": [ira_est, ira_se],
    "CRA w Lasso": [cra_lasso_est, cra_lasso_se],
    "IRA w Lasso": [ira_lasso_est, ira_lasso_se],
}
results = pd.DataFrame(results)
results.index = ["Estimate", "Standard Error"]
results
# for printing to LaTeX: print(results.to_latex())

,CL,CRA,IRA,CRA w Lasso,IRA w Lasso
Estimate,-0.085455,-0.079680,-1.952213e+08,-0.079329,-0.071260
Standard Error,0.035849,0.035226,2.445273e+08,0.035383,0.035308


Treatment group 4 experiences an average decrease of about $7.8\%$ in the length of unemployment spell.


Observe that regression estimators delivers estimates that are slighly more efficient (lower standard errors) than the simple 2 mean estimator, but essentially all methods have very similar standard errors.  We also see the regression estimators offer slightly lower estimates -- these difference occur perhaps to due minor imbalance in the treatment allocation, which the regression estimators try to correct.


